# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divinely06/CloneFlyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Refresh / Content Opportunity Scoring.**

I chose this lane because the starter dataset shows which pages have enough demand to matter, and the output (a ranked review queue) maps directly to a real decision: which page a reviewer looks at first. It also leaves room to grow, since I can later replace the proxy label with a future-window label on the warehouse data. I may revisit this lane by the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

- **Search question:** Which pages should a reviewer look at first for refresh, protection, or monitoring, given limited review time?
- **Unit of analysis:** one content page (one row per `content_id`) over a 90-day window.
- **Output:** a ranked review queue with a score and reason codes for each page.
- **Action:** a content reviewer checks the top K pages each cycle (for example 20 or 50) and decides whether to refresh, protect, or monitor.
- **Cost of a wrong call:** a false positive wastes reviewer time on a page that was fine. A false negative means a page that was losing visibility goes unnoticed for another cycle. Which error is worse depends on review capacity, so I will treat it as a policy choice and write down what I chose.
- **Why data or ML can help:** a person cannot eyeball thousands of pages across impressions, position, CTR, engagement, and age. A transparent baseline score can rank them, and a learned model may rank them better. I will test that against the baseline rather than assume it.
- **Why this is not just "train a model":** most of the work happens before any model exists. I have to define what "decline" means (magnitude, window, persistence, minimum volume), build a simple baseline to beat, avoid leakage, and validate on clients the model has not seen. I will judge the ranking by precision@K, not accuracy.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
import pandas as pd
# upload data/raw/content_refresh_anonymized.csv to /content first
df = pd.read_csv("/content/CloneFlyrank/data/raw/content_refresh_anonymized.csv")

# Use the skeleton's own loading line for df, then:
print("raw rows:", len(df), "| clients:", df["client_id"].nunique())

# Same filters the starter pipeline uses
d = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
print("rows after starter filters:", len(d))

# Number 1: how common is the proxy 'declining' label?
print("share trend_direction == 'down':", round((d["trend_direction"] == "down").mean(), 3))

# Number 2: how concentrated is visibility?
imp = d["impressions_90d"].sort_values(ascending=False)
print("share of impressions held by top 10% of pages:", round(imp.head(int(len(imp) * 0.10)).sum() / imp.sum(), 3))

# Number 3: how many pages look like CTR review candidates? (check ctr units first)
cand = d[(d["impressions_90d"] >= 500) & (d["avg_position"].between(1, 20)) & (d["ctr"] < 0.5)]
print("CTR review candidates:", len(cand), "of", len(d))

raw rows: 30000 | clients: 32
rows after starter filters: 30000
share trend_direction == 'down': 0.542
share of impressions held by top 10% of pages: 0.702
CTR review candidates: 9745 of 30000


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** in this starter sample of 30,000 pages from 32 clients, 54% carry a "down" trend label, and impressions are heavily concentrated in a small share of pages. These are observed, directional, decision-support findings.

**What I can't say:** that a refresh causes recovery, that results hold on the full warehouse, or that `trend_direction == "down"` is a true future outcome (it is a proxy computed from the current window). I also can't claim anything about Google's algorithm.

- 54.2% of pages are labeled "down". That is common, not rare, so I will judge my ranking by precision@K against that base rate.
- The top 10% of pages hold 70.2% of impressions. Visibility is concentrated, so I will use a minimum-volume filter.
- 9,745 of 30,000 pages match the CTR review rule. That is too many for a person to check, so I need a ranked list with reasons.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.